In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm import tqdm

from collections import Counter
from collections import defaultdict

from albumentations.pytorch import ToTensorV2
import albumentations as A

import cv2
import numpy as np
import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

import random
import os
from glob import glob

d:\LabsMIET\MLlab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 9999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
class_to_idx = { "Апельсин": 0,
                 "Бананы": 1,
                 "Груши": 2, 
                 "Кабачки": 3, 
                 "Капуста": 4, 
                 "Картофель": 5, 
                 "Киви": 6, 
                 "Лимон": 7, 
                 "Лук": 8, 
                 "Мандарины": 9, 
                 "Морковь": 10, 
                 "Огурцы": 11, 
                 "Томаты": 12, 
                 "Яблоки зелёные": 13, 
                 "Яблоки красные": 14 }

In [4]:
class MyDataset(Dataset):
    def __init__(self, images_filepaths, name2label, transform=None):
        self.images_filepaths = images_filepaths
        self.transform = transform
        self.name2label = name2label

    def __len__(self):
        return len(self.images_filepaths)

    def __getitem__(self, idx):
        image_filepath = self.images_filepaths[idx]
        image = cv2.imdecode(np.fromfile(image_filepath, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.name2label[os.path.normpath(image_filepath).split(os.sep)[-3]]
        
        if self.transform is not None:
            image = self.transform(image=image)['image']
        return image, label


def train_test_split_from_directory(root_path, folder2class, train_size=0.8):
    train, test = [], []

    for class_name in os.listdir(root_path):
        class_path = os.path.join(root_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for subclass_name in os.listdir(class_path):
            subclass_path = os.path.join(class_path, subclass_name)
            if not os.path.isdir(subclass_path):
                continue

            images = glob(os.path.join(subclass_path, '*.jpg')) + \
                     glob(os.path.join(subclass_path, '*.png')) + \
                     glob(os.path.join(subclass_path, '*.jpeg'))
            
            if len(images) == 0:
                continue
            
            # делим подклассы в пропорции 80/20
            random.shuffle(images)
            split_idx = int(train_size * len(images))

            if split_idx == 0 and len(images) > 0:
                split_idx = 1

            train.extend(images[:split_idx])
            test.extend(images[split_idx:])

    random.shuffle(train)
    random.shuffle(test)

    return train, test

Настройка датасета, writer для вывода, device - на чем обучается

In [5]:
dataset_path = 'train/train'
train, test = train_test_split_from_directory(dataset_path, class_to_idx)

writer = SummaryWriter("kirillLogs")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Функция ошибки

In [6]:
class FocalSmoothingLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits, targets):
        n_classes = logits.size(-1)
        
        # Label smoothing  
        with torch.no_grad():
            true_dist = torch.full_like(logits, self.label_smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        
        # Log softmax 
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Cross entropy с label smoothing
        ce_loss = -(true_dist * log_probs).sum(dim=-1)        
        
        # Focal-часть 
        pt = torch.exp(-ce_loss)                                 
        modulating_factor = (1 - pt) ** self.gamma
        
        #  Class weights 
        if self.alpha is not None:
            if self.alpha.dim() == 1:  
                alpha_t = self.alpha[targets]
            else:
                alpha_t = self.alpha
            loss = alpha_t * modulating_factor * ce_loss
        else:
            loss = modulating_factor * ce_loss
        
        # Редукция
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

Модель

In [ ]:
class HierarchicalSwinV2(nn.Module):
    def __init__(self, num_classes=15, model_name="swinv2_cr_tiny_ns_224.sw_in1k"):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True  
        )

        feature_channels = self.backbone.feature_info.channels()

        self.heads = nn.ModuleList([
            nn.Sequential(  # stage 1 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[0], 96),
                nn.ReLU(),
            ),
            nn.Sequential(  # stage 2
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[1], 128),
                nn.ReLU(),
            ),
            nn.Sequential(  # stage 3 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[2], 192),
                nn.ReLU(),
                nn.Dropout(0.2),
            ),
            nn.Sequential(  # stage 4 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[3], 256),
                nn.ReLU(),
                nn.Dropout(0.3),
            ),
        ])
        
        self.classifier = nn.Sequential(
            nn.Linear(672, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x) 
    
        pooled = []
        for i in range(4):
            feat = features[i]
            out = self.heads[i](feat)        
            pooled.append(out)
    
        fused = torch.cat(pooled, dim=1)     
    
        logits = self.classifier(fused)
        return logits

In [8]:
# class HierarchicalSwinV2(nn.Module):
#     def __init__(self, num_classes=15, model_name="swinv2_cr_small_ns_224.sw_in1k"):
#         super().__init__()

#         self.backbone = timm.create_model(
#             model_name,
#             pretrained=True,
#             features_only=True  
#         )

#         feature_channels = self.backbone.feature_info.channels()

#         self.heads = nn.ModuleList([
#             nn.Sequential(  # stage 1: Deeper MLP
#                 nn.AdaptiveAvgPool2d(1), 
#                 nn.Flatten(),
#                 nn.Linear(feature_channels[0], 128),  # Increased dim
#                 nn.ReLU(),
#                 nn.Dropout(0.1),
#                 nn.Linear(128, 96),
#                 nn.ReLU(),
#             ),
#             nn.Sequential(  # stage 2
#                 nn.AdaptiveAvgPool2d(1), 
#                 nn.Flatten(),
#                 nn.Linear(feature_channels[1], 192),  # Increased dim
#                 nn.ReLU(),
#                 nn.Dropout(0.15),
#                 nn.Linear(192, 128),
#                 nn.ReLU(),
#             ),
#             nn.Sequential(  # stage 3 
#                 nn.AdaptiveAvgPool2d(1), 
#                 nn.Flatten(),
#                 nn.Linear(feature_channels[2], 256),
#                 nn.ReLU(),
#                 nn.Dropout(0.2),
#                 nn.Linear(256, 192),
#                 nn.ReLU(),
#             ),
#             nn.Sequential(  # stage 4 with attention example
#                 nn.AdaptiveAvgPool2d(1), 
#                 nn.Flatten(),
#                 nn.Linear(feature_channels[3], 512),
#                 nn.ReLU(),
#                 nn.Dropout(0.2),
#                 nn.Linear(512, 256),  # Intermediate
#                 nn.ReLU(),
#                 nn.Dropout(0.2),
#                 nn.Linear(256, 256),  # Output
#             ),
#         ])
        
#         concat_dim = 96 + 128 + 192 + 256  # Adjust based on head outputs
#         self.fusion_attn = nn.MultiheadAttention(embed_dim=concat_dim, num_heads=8)
        
#         self.classifier = nn.Sequential(
#             nn.Linear(concat_dim, 512),  # Increased hidden dim
#             nn.ReLU(),
#             nn.Dropout(0.4),  # Tuned dropout
#             nn.Linear(512, 256),
#             nn.ReLU(),
#             nn.Dropout(0.4),
#             nn.Linear(256, num_classes)
#         )

#     def forward(self, x):
#         features = self.backbone(x) 
    
#         pooled = []
#         for i in range(4):
#             feat = features[i]
#             out = self.heads[i](feat)        
#             pooled.append(out)
    
#         fused = torch.cat(pooled, dim=1)     
#         fused = fused.unsqueeze(0)  # [1, B, concat_dim]
#         fused, _ = self.fusion_attn(fused, fused, fused)
#         fused = fused.squeeze(0)
        
#         logits = self.classifier(fused)
#         return logits

Классы, классы весов, sampling

Аугментация

In [9]:
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.75, 1.0)),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),

    A.Affine(
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
        scale=(0.9, 1.1),      
        rotate=(-100, 100),          
        border_mode=0,
        value=0,
        p=0.85
    ),

    A.CLAHE(p=0.1),

    A.ColorJitter( # Гамма
        brightness=0.25,
        contrast=0.25, 
        saturation=0.25, 
        hue=0.01, 
        p=0.5
    ),

    A.RandomShadow(p=0.25), # Тени
    A.RandomFog(p=0.15, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман

    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), sigma_limit=(0.1, 1.5)),
        A.GaussNoise(var_limit=(5.0, 30.0)),
    ], p=0.8),

    A.CoarseDropout(
        num_holes_range=(1, 6),
        hole_height_range=(0.03, 0.12),
        hole_width_range=(0.03, 0.12),
        fill=128,
        p=0.7
    ),

    A.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

C:\Users\Kirill\AppData\Local\Temp\ipykernel_28268\619999477.py:7: UserWarning: Argument(s) 'value' are not valid for transform Affine
  A.Affine(
C:\Users\Kirill\AppData\Local\Temp\ipykernel_28268\619999477.py:27: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(p=0.15, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман
C:\Users\Kirill\AppData\Local\Temp\ipykernel_28268\619999477.py:31: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0)),


Датасеты, лоадеры

In [10]:
train_dataset = MyDataset(
    images_filepaths=train, 
    name2label=class_to_idx, 
    transform=train_transforms
)
test_dataset = MyDataset(
    images_filepaths=test, 
    name2label=class_to_idx, 
    transform=val_transforms
)

In [11]:
def count_classes(image_paths, class_to_idx):
    counts = Counter()
    for path in image_paths:
        class_name = os.path.normpath(path).split(os.sep)[-3]
        class_idx = class_to_idx[class_name]
        counts[class_idx] += 1
    return counts

train_counts = count_classes(train, class_to_idx)
test_counts  = count_classes(test, class_to_idx)

total_samples = sum(train_counts.values())         
num_classes = len(train_counts)

class_weights = torch.tensor(
    [total_samples / (num_classes * train_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
)

class_weight_arr_freq = class_weights / class_weights.mean()

# train_labels = torch.tensor([label for _, label in train_dataset])

# weigth_sampling = class_weights[train_labels]

# sampler_weigths = WeightedRandomSampler(
#     weights=weigth_sampling,
#     num_samples=len(weigth_sampling),
#     replacement=True
# )


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    # sampler=sampler_weigths,
    shuffle=True,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False  
)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False
)

Функция для обучения

Инициализация модели

Обучение

Тест

In [13]:
@torch.no_grad()
def evaluate(model, dataloader, loss_fn, device, desc="Val"):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc=desc, leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        y_pred = logits.argmax(dim=1)
        total_correct += (y_pred == labels).sum().item()
        total_samples += batch_size

        avg_loss = total_loss / max(total_samples, 1)
        acc = total_correct / max(total_samples, 1)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{acc:.4f}")

    avg_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)
    return accuracy, avg_loss

In [14]:
# def training_with_stage(model, criterion, train_loader, val_loader, device, writer, n_epoch=30):
#     num_iter = 0
#     best_val_acc = 0.0
#     best_val_loss = 100
#     counter_early_stop = 0
    
#     print('Stage 1', n_epoch)

#     for param in model.parameters(): 
#         param.requires_grad = False

#     for n, p in model.named_parameters():
#         if "head" in n: p.requires_grad = True

#     head_params = [p for n, p in model.named_parameters() if "head" in n]

#     optimizer_stage1 = optim.AdamW(head_params, lr=1e-3, weight_decay=0.05)

#     scheduler_stage1 = torch.optim.lr_scheduler.CosineAnnealingLR(
#         optimizer_stage1,
#         T_max=5 * len(train_loader),   # 5 эпох
#         eta_min=1e-6
#     )

#     for epoch in range(1, 6):
#         model.train()

#         total_loss = 0.0
#         total_correct = 0
#         total_samples = 0

#         pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epoch}", leave=True)

#         for images, labels in pbar:
#             images = images.to(device)
#             labels = labels.to(device)

#             logits = model(images)
#             loss = criterion(logits, labels)

#             optimizer_stage1.zero_grad(set_to_none=True)
#             loss.backward()
            
#             optimizer_stage1.step()
#             scheduler_stage1.step()
#             batch_size = labels.size(0)
#             total_loss += loss.item() * batch_size
#             total_samples += batch_size

#             y_pred = logits.argmax(dim=1)
#             total_correct += (y_pred == labels).sum().item()

#             avg_loss = total_loss / max(total_samples, 1)
#             acc = total_correct / max(total_samples, 1)

#             # tqdm live-metrics
#             pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

#             # Логирование (по итерациям)
#             num_iter += 1
#             if writer is not None:
#                 writer.add_scalar("Loss/train", loss.item(), num_iter)
#                 writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

#         # Валидация (тоже с tqdm)
#         val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epoch}")

#         if writer is not None:
#             writer.add_scalar("Loss/val", val_loss, num_iter)
#             writer.add_scalar("Accuracy/val", val_acc, num_iter)

#         print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

#         # Проверка на лучшую модель (для Stage 1 тоже добавляем, чтобы сохранять прогресс)
#         if val_acc > best_val_acc:
#             best_val_acc = val_acc
#             best_val_loss = val_loss
#             counter_early_stop = 0

#             save_path = f"checkpoints/best_model.pth"
#             torch.save(model.state_dict(), save_path)
#             print(f"--- Эпоха {epoch}: Новая лучшая точность: {val_acc:.4f}! Модель сохранена. ---")
#         elif (val_acc == best_val_acc) and (val_loss < best_val_loss):
#             best_val_loss = val_loss
#             counter_early_stop = 0

#             save_path = f"checkpoints/best_model.pth"
#             torch.save(model.state_dict(), save_path)
#             print(f"--- Эпоха {epoch}: Новая лучшая точность: {val_acc:.4f}! Модель сохранена. ---")
#         else:
#             counter_early_stop += 1
#             if counter_early_stop > 5:
#                 print(f"Сработал Early stop!")
#                 break
    
#     del optimizer_stage1, scheduler_stage1

#     print('Stage 2')
    
#     # Размораживаем все параметры для fine-tuning
#     for n, p in model.named_parameters():
#         if "head" not in n: p.requires_grad = True

#     # Разделяем параметры для разного LR
#     backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
#     head_params = [p for n, p in model.named_parameters() if "head" in n]

#     optimizer_stage2 = optim.AdamW([
#         {'params': backbone_params, 'lr': 3e-6, 'weight_decay': 0.01},
#         {'params': head_params, 'lr': 3e-5, 'weight_decay': 0.05},
#     ])

#     scheduler_stage2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#         optimizer_stage2,
#         T_0=(n_epoch - 5) * len(train_loader),   # оставшиеся эпохи
#         eta_min=1e-7
#     )

#     for epoch in range(6, n_epoch + 1):
#         model.train()

#         total_loss = 0.0
#         total_correct = 0
#         total_samples = 0

#         pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epoch}", leave=True)

#         for images, labels in pbar:
#             images = images.to(device)
#             labels = labels.to(device)

#             logits = model(images)
#             loss = criterion(logits, labels)

#             optimizer_stage2.zero_grad(set_to_none=True)
#             loss.backward()
            
#             optimizer_stage2.step()
#             scheduler_stage2.step()

#             batch_size = labels.size(0)
#             total_loss += loss.item() * batch_size
#             total_samples += batch_size

#             y_pred = logits.argmax(dim=1)
#             total_correct += (y_pred == labels).sum().item()

#             avg_loss = total_loss / max(total_samples, 1)
#             acc = total_correct / max(total_samples, 1)

#             # tqdm live-metrics
#             pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

#             # Логирование (по итерациям)
#             num_iter += 1
#             if writer is not None:
#                 writer.add_scalar("Loss/train", loss.item(), num_iter)
#                 writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

#         # Валидация (тоже с tqdm)
#         val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epoch}")

#         if writer is not None:
#             writer.add_scalar("Loss/val", val_loss, num_iter)
#             writer.add_scalar("Accuracy/val", val_acc, num_iter)

#         print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

#         # Проверка на лучшую модель
#         if val_acc > best_val_acc:
#             best_val_acc = val_acc
#             best_val_loss = val_loss
#             counter_early_stop = 0

#             save_path = f"checkpoints/best_model.pth"
#             torch.save(model.state_dict(), save_path)
#             print(f"--- Эпоха {epoch}: Новая лучшая точность: {val_acc:.4f}! Модель сохранена. ---")
#         elif (val_acc == best_val_acc) and (val_loss < best_val_loss):
#             best_val_loss = val_loss
#             counter_early_stop = 0

#             save_path = f"checkpoints/best_model.pth"
#             torch.save(model.state_dict(), save_path)
#             print(f"--- Эпоха {epoch}: Новая лучшая точность: {val_acc:.4f}! Модель сохранена. ---")
#         else:
#             counter_early_stop += 1
#             if counter_early_stop > 5:
#                 print(f"Сработал Early stop!")
#                 break

#     # Загружаем лучший вес в конце (как в примере)
#     if os.path.exists("checkpoints/best_model.pth"):
#         model.load_state_dict(torch.load("checkpoints/best_model.pth"))
#         os.remove("checkpoints/best_model.pth")
#     del optimizer_stage2, scheduler_stage2
#     return model

In [ ]:
def train(model, loss_fn, optimizer, train_loader, val_loader, device, writer=None, n_epoch=3, lr_eta=1e-6):
    best_val_acc = 0.0
    best_val_loss = 100
    num_iter = 0
    counter_early_stop = 0

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epoch, eta_min=lr_eta) # планировщик скорости обучения

    for epoch in range(1, n_epoch + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epoch}", leave=True)

        for X_batch, y_batch in pbar:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            # накопим метрики для прогресс-бара
            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == y_batch).sum().item()

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # tqdm live-metrics
            pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

            # логирование (по итерациям)
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", (y_pred == y_batch).float().mean().item(), num_iter)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        # Валидация (тоже с tqdm)
        val_acc, val_loss = evaluate(model, val_loader, loss_fn, device, desc=f"Val {epoch}/{n_epoch}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)
            writer.add_scalar(f"LR", current_lr, epoch)

        print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        # Проверка на лучшую модель
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            counter_early_stop = 0

            save_path = f"checkpointsSwin/best_model.pth"
            torch.save(model.state_dict(), save_path)
            print(f"--- Эпоха {epoch}: Новая лучшая точность: {val_acc:.4f}! Модель сохранена. ---")
        elif (val_acc == best_val_acc) and (val_loss < best_val_loss):
            best_val_loss = val_loss
            counter_early_stop = 0

            save_path = f"checkpointsSwin/best_model.pth"
            torch.save(model.state_dict(), save_path)
            print(f"--- Эпоха {epoch}: Новая лучшая точность: {val_acc:.4f}! Модель сохранена. ---")
        else: # Early stop
            counter_early_stop += 1
            if counter_early_stop > 5:
                print(f"Сработал Earle stop!")
                break

    # Загружаем лучший вес
    model.load_state_dict(torch.load(f"checkpointsSwin/best_model.pth"))
    os.remove(f"checkpointsSwin/best_model.pth")
    return model

In [ ]:
os.makedirs('checkpointsSwin', exist_ok=True)
n_epochs = 20

class_weights = class_weights.to(device)

model = HierarchicalSwinV2(num_classes=15).to(device)

# model_name = "swinv2_cr_tiny_ns_224.sw_in1k"
# model = timm.create_model(model_name, pretrained=True, num_classes=15).to(device)

# criterion = nn.CrossEntropyLoss(
#     weight=class_weights,
#     label_smoothing=0.05,      
#     reduction='mean'
# )

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

criterion = FocalSmoothingLoss(
    alpha=class_weights,
    gamma=2.0,
    label_smoothing=0.08,
    reduction='mean'
)

In [ ]:
model = train(
    model,
    criterion,
    optimizer,
    train_loader,
    test_loader,
    device,
    writer,
    n_epochs
)

In [18]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def sklearn_report(model, dataloader, device, idx2class=None, digits=4):
    model.eval()

    y_true, y_pred = [], []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)

        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_pred.append(preds)
        y_true.append(labels.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    if idx2class is None:
        target_names = None
        labels = None
    else:
        labels = sorted(idx2class.keys())
        target_names = [idx2class[i] for i in labels]

    rep = classification_report(
        y_true, y_pred,
        labels=labels,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )
    print(rep)

    if idx2class is not None:
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print("\nConfusion Matrix:")
        print(cm)

In [19]:
idx2class = {v: k for k, v in class_to_idx.items()}

sklearn_report(model, test_loader, device, idx2class=idx2class, digits=4)

                precision    recall  f1-score   support

      Апельсин     0.8836    0.9435    0.9126       177
        Бананы     0.8239    0.8951    0.8580       162
         Груши     0.8182    0.8471    0.8324        85
       Кабачки     0.5104    0.7778    0.6164        63
       Капуста     0.9299    0.8690    0.8985       168
     Картофель     0.8897    0.8165    0.8515       158
          Киви     0.6667    0.8085    0.7308        47
         Лимон     0.9302    0.9023    0.9160       133
           Лук     0.8500    0.7727    0.8095       132
     Мандарины     0.9189    0.8718    0.8947       156
       Морковь     0.8770    0.8629    0.8699       124
        Огурцы     0.9127    0.9829    0.9465       117
        Томаты     0.9051    0.9533    0.9286       150
Яблоки зелёные     0.8489    0.6941    0.7638       170
Яблоки красные     0.8116    0.7671    0.7887       146

      accuracy                         0.8546      1988
     macro avg     0.8385    0.8510    0.8412 

In [22]:
test_images_dir = "test_images/test_images"
submission_path = "sample_submission.csv"
output_path = "submissionKirill.csv"

In [23]:
import pandas as pd
submission = pd.read_csv(submission_path)

model.eval()
pred_labels = []

with torch.no_grad():
    for image_id in tqdm(submission["image_id"], desc="Predicting"):
        image_path = os.path.join(test_images_dir, image_id)

        image = cv2.imdecode(
            np.fromfile(image_path, dtype=np.uint8),
            cv2.IMREAD_COLOR
        )
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Когда мы хотим сделать предсказание, нам нужно знать, а какие были преобразования при обучении/тестировании
        if val_transforms is not None:
            image = val_transforms(image=image)["image"]

        image = image.unsqueeze(0).to(device)

        logits = model(image)
        pred_idx = logits.argmax(dim=1).item()

        pred_labels.append(pred_idx)


Predicting: 100%|██████████| 2503/2503 [01:06<00:00, 37.82it/s]


In [24]:
submission["label"] = pred_labels
submission.to_csv(output_path, index=False)

submission.head()


,image_id,label
0,fd343552326b42c5a62c192f32549dc7.jpg,2
1,445ca69812cf44f581cc8a89223af277.jpg,7
2,570626ce4d8f41edb8088f49d40a2195.jpg,7
3,02d4acba92f343d798adcb4958fe684b.jpg,11
4,2d4b8e8f38534a39b0d02c440e917b83.jpg,12
